# Matchup engine (Stage 4)

Join a BGC-Argo mixed-layer summary to a PACE granule and write the matchup
record linking **float ↔ granule ↔ pixels**. This notebook works entirely on
**synthetic** data (no network); the criteria and selection rule are documented
on the [Matchup engine](../matchup.rst) page.

## 1. A synthetic granule

The canonical granule layout (dims `(x, y, wl)`; 2-D `latitude`/`longitude`; `l2_flags`) is what `pab.pace.cloud.open_granule` produces. Here we build a tiny 5×5×4 one centered on a float position.

In [1]:
import numpy as np
import xarray as xr
from pab.pace import flags

def make_granule(center=(20.0, -50.0), span=0.5, flagged=()):
    nx, ny, nw = 5, 5, 4
    clat, clon = center
    lat = np.linspace(clat - span, clat + span, nx)
    lon = np.linspace(clon - span, clon + span, ny)
    lons2d, lats2d = np.meshgrid(lon, lat)
    wave = np.array([440.0, 490.0, 550.0, 670.0])
    rrs = np.fromfunction(lambda i, j, k: 0.01 * (i * 10 + j + 1), (nx, ny, nw))
    l2 = np.zeros((nx, ny), dtype=np.int64)
    for ix, iy, name in flagged:
        l2[ix, iy] |= flags.flag_value([name])
    return xr.Dataset(
        {"Rrs": (("x", "y", "wl"), rrs),
         "Rrs_unc": (("x", "y", "wl"), rrs * 0.1),
         "l2_flags": (("x", "y"), l2)},
        coords={"latitude": (("x", "y"), lats2d),
                "longitude": (("x", "y"), lons2d),
                "wavelength": ("wl", wave)})

granule = make_granule()
granule

<xarray.Dataset> Size: 2kB
Dimensions:     (x: 5, y: 5, wl: 4)
Coordinates:
    latitude    (x, y) float64 200B 19.5 19.5 19.5 19.5 ... 20.5 20.5 20.5 20.5
    longitude   (x, y) float64 200B -50.5 -50.25 -50.0 ... -50.0 -49.75 -49.5
    wavelength  (wl) float64 32B 440.0 490.0 550.0 670.0
Dimensions without coordinates: x, y, wl
Data variables:
    Rrs         (x, y, wl) float64 800B 0.01 0.01 0.01 0.01 ... 0.45 0.45 0.45
    Rrs_unc     (x, y, wl) float64 800B 0.001 0.001 0.001 ... 0.045 0.045 0.045
    l2_flags    (x, y) int64 200B 0 0 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0 0

## 2. Seed a profile and a granule in the store

The matchup engine reads qualifying profiles (those with a mixed-layer summary) and the `granules` candidate set from the DB, then writes the matchup back. We seed a tiny in-memory store.

In [2]:
from pab.db import Store
from pab.argo.summary import persist_summary

store = Store.open(":memory:")

profile_id = persist_summary(
    store, wmo=7902226, cycle=5,
    summary={"mld": 35.0, "mld_method": "deBoyerMontegut_0.03", "n_points": 8},
    latitude=20.0, longitude=-50.0, time="2025-05-01T12:00:00",
)
store.upsert("granules", {
    "granule_id": "PACE_OCI.20250501T1130.L2.OC_AOP",
    "time_start": "2025-05-01T11:30:00",
    "data_url": "s3://synthetic/granule.nc",
})
profile_id

1

## 3. Build the matchup (offline)

We inject the synthetic granule through the `opener=` seam, so no S3/network is touched. `build_matchups` selects the best granule per profile and persists the record.

In [3]:
from pab.matchup import engine

result = engine.build_matchups(store, opener=lambda src: granule)
result

{'written': ['7902226_5_PACE_OCI.20250501T1130.L2.OC_AOP'],
 'skipped': [],
 'unmatched': []}

## 4. The selected pixels and written records

In [4]:
import pandas as pd
matchups = store.query_df("SELECT * FROM matchups")
matchups

,matchup_id,profile_id,granule_id,distance_km,dtime_hours,n_spectra,created,pab_version
0,7902226_5_PACE_OCI.20250501T1130.L2.OC_AOP,1,PACE_OCI.20250501T1130.L2.OC_AOP,0.0,0.5,10,2026-06-20T22:00:44.284166+00:00,0.0.dev0


In [5]:
pixels = store.query_df(
    "SELECT ix, iy, latitude, longitude, distance_km, rank, flagged "
    "FROM matchup_pixels ORDER BY rank")
pixels

,ix,iy,latitude,longitude,distance_km,rank,flagged
0,2,2,20.00,-50.00,0.000000,1,0
1,2,3,20.00,-49.75,26.122297,2,0
2,2,1,20.00,-50.25,26.122297,3,0
3,1,2,19.75,-50.00,27.798770,4,0
4,3,2,20.25,-50.00,27.798770,5,0
5,3,3,20.25,-49.75,38.132112,6,0
6,3,1,20.25,-50.25,38.132112,7,0
7,1,3,19.75,-49.75,38.160521,8,0
8,1,1,19.75,-50.25,38.160521,9,0
9,2,0,20.00,-50.50,52.244579,10,0


The ~10 nearest **unflagged** pixels were chosen, nearest first (`rank` 1 = nearest), all `flagged = 0`. The `matchups` row records `distance_km` (float → nearest pixel), `dtime_hours`, `n_spectra`, and provenance.

## 5. Idempotent re-run

The deterministic `matchup_id` means a re-run upserts rather than duplicates — it is skipped and the row counts are unchanged.

In [6]:
rerun = engine.build_matchups(store, opener=lambda src: granule)
print("written:", rerun["written"])
print("skipped:", rerun["skipped"])
print("matchups rows :", store.count("matchups"))
print("pixel rows    :", store.count("matchup_pixels"))

written: []
skipped: ['7902226_5_PACE_OCI.20250501T1130.L2.OC_AOP']
matchups rows : 1
pixel rows    : 10


## 6. A flagged nearest pixel is excluded

Flag the pixel under the float; the engine selects the nearest *unflagged* neighbour instead. (The synthetic grid is coarse — ~26 km/pixel — so we loosen the distance gate for the demo.)

In [7]:
flagged_granule = make_granule(flagged=[(2, 2, "LAND")])
prof = {"wmo": 7902226, "cycle": 5, "latitude": 20.0, "longitude": -50.0,
        "time": "2025-05-01T12:00:00", "profile_id": profile_id}
cands = [{"granule_id": "G_flag", "time": "2025-05-01T12:00:00", "source": "x"}]
cfg = engine.MatchupConfig(max_distance_km=50.0)
m = engine.find_matchup(prof, cands, opener=lambda src: flagged_granule, config=cfg)
print("nearest selected pixel (ix, iy):", (m.pixels[0]["ix"], m.pixels[0]["iy"]))
print("pixel (2,2) selected? ", any((p["ix"], p["iy"]) == (2, 2) for p in m.pixels))

nearest selected pixel (ix, iy): (2, 3)
pixel (2,2) selected?  False


## 7. (Optional) live matchup

Set `RUN_LIVE = True` to match a **real** float profile (argopy) to a discovered PACE granule
(earthaccess). This needs network access and an Earthdata Login; it is **off** by default so the
notebook builds offline. The dev profile set lives in `data/dev_profiles.csv`.

In [8]:
RUN_LIVE = False  # flip to True (needs network + Earthdata Login)

if RUN_LIVE:
    import pandas as pd
    from datetime import datetime, timedelta
    from pab.argo import fetch
    from pab.pace import discover, cloud

    dev = pd.read_csv("../../data/dev_profiles.csv")
    seed = dev[dev.known_matchup == 1].iloc[0]
    wmo, cycle = int(seed.wmo), int(seed.cycle)

    # 1. fetch the float profile, summarize, persist
    ds = fetch.fetch_profile(wmo, cycle)
    meta, vrs = next(fetch.iter_profiles(ds))
    # 2. discover PACE granules in a small box / +-1 day window
    t = datetime.fromisoformat(meta["time"])
    win = (t - timedelta(days=1), t + timedelta(days=1))
    lon, lat = meta["longitude"], meta["latitude"]
    results = discover.search_granules(
        temporal=(win[0].isoformat(), win[1].isoformat()),
        bounding_box=(lon - 0.5, lat - 0.5, lon + 0.5, lat + 0.5))
    print(f"{len(results)} candidate granules for {wmo}/{cycle}")
    # 3. open the nearest granule and build the matchup
    #    (open_granule(url, backend="s3") in us-west-2), then engine.find_matchup(...)
